In [0]:
df = spark.read.json("/Volumes/stock/default/stock_volume/stock_data.json")
display(df)

price,stock,timestamp
1348.8,RELIANCE,1.7746042691035876E9
1348.8,RELIANCE,1.7746042746584523E9
1348.4,RELIANCE,1.7746042801782212E9
1348.4,RELIANCE,1.7746042856648405E9
1348.2,RELIANCE,1.7746042912460928E9
1348.2,RELIANCE,1.7746042967605045E9
1348.8,RELIANCE,1.774604302325496E9
1348.8,RELIANCE,1.7746043084189446E9
1348.9,RELIANCE,1.774604314491018E9
1348.9,RELIANCE,1.7746043205164213E9


### Convert Timestamp

In [0]:
from pyspark.sql.functions import from_unixtime, col

df = df.withColumn("time", from_unixtime(col("timestamp")))



### Window Function

In [0]:
from pyspark.sql.window import Window

window = Window.partitionBy("stock").orderBy("timestamp")

### Change Percentage

In [0]:
from pyspark.sql.functions import lag, round

df = df.withColumn("prev_price", lag("price").over(window))

df = df.withColumn(
    "percent_change",
    round((col("price") - col("prev_price")) / col("prev_price") * 100, 2)
)

###Calculating Moving Averages

In [0]:
from pyspark.sql.functions import avg

window_short = Window.partitionBy("stock").orderBy("timestamp").rowsBetween(-5, 0)
window_long = Window.partitionBy("stock").orderBy("timestamp").rowsBetween(-20, 0)

df = df.withColumn("ma_short", round(avg("price").over(window_short), 2))
df = df.withColumn("ma_long", round(avg("price").over(window_long), 2))

### Volatility

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import stddev, round

window_vol = Window.partitionBy("stock")

df = df.withColumn(
    "volatility",
    round(stddev("price").over(window_vol), 2)
)

### Buy/Sell Signal

In [0]:
from pyspark.sql.functions import when

df = df.withColumn(
    "signal",
    when(col("ma_short") > col("ma_long"), "BUY")
    .when(col("ma_short") < col("ma_long"), "SELL")
    .otherwise("HOLD")
)

### Display Table

In [0]:
df = df.select(
    "stock",
    "price",
    "time",
    "percent_change",
    "ma_short",
    "ma_long",
    "volatility",
    "signal"
)

display(df)

stock,price,time,percent_change,ma_short,ma_long,volatility,signal
AAPL,252.89,2026-03-27 10:34:41,null,252.89,252.89,0.0,HOLD
AAPL,252.89,2026-03-27 10:34:55,0.0,252.89,252.89,0.0,HOLD
AAPL,252.89,2026-03-27 10:35:08,0.0,252.89,252.89,0.0,HOLD
AAPL,252.89,2026-03-27 10:35:21,0.0,252.89,252.89,0.0,HOLD
AAPL,252.89,2026-03-27 10:35:34,0.0,252.89,252.89,0.0,HOLD
AAPL,252.89,2026-03-27 10:35:47,0.0,252.89,252.89,0.0,HOLD
AAPL,252.89,2026-03-27 10:36:00,0.0,252.89,252.89,0.0,HOLD
AAPL,252.89,2026-03-27 10:36:14,0.0,252.89,252.89,0.0,HOLD
AAPL,252.89,2026-03-27 10:36:27,0.0,252.89,252.89,0.0,HOLD
AAPL,252.89,2026-03-27 10:36:40,0.0,252.89,252.89,0.0,HOLD
